In [1]:
import os
import json
import sys
from dotenv import load_dotenv

os.chdir("C:/Files/SeMedia/backend")
load_dotenv()

# Clear cached modules
for mod in list(sys.modules.keys()):
    if 'tests' in mod or 'orchestrator' in mod:
        del sys.modules[mod]

In [2]:
from tests.orchestrator.gen_data import EvaluationDatasetGenerator
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas import evaluate

from ragas.metrics import (
    answer_correctness,
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

from tests.orchestrator.schema import EvaluationDatasetGeneratorSchema

c:\Files\SeMedia\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Ziad\AppData\Local\Temp\ipykernel_36308\1143636511.py:5: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\Ziad\AppData\Local\Temp\ipykernel_36308\1143636511.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Ziad\AppData\Local\Temp\ipykernel_36308\1143636511.py:5: DeprecationWarning: Importing faithfuln

In [3]:
async def save_data():
    generator = EvaluationDatasetGenerator()
    data_file_path = os.path.join("tests", "data", "eval_data.json")
    
    if os.path.exists(data_file_path) and os.path.getsize(data_file_path) > 0:
        with open(data_file_path, 'r') as json_file:
            eval_data = json.load(json_file)
    else:
        eval_data = []
    
    x = 14 - len(eval_data)
    for i in range(x):
        record = await generator.generate_data()
        eval_data.append(record)
         
        with open(data_file_path, 'w') as json_file:
            json.dump(eval_data, json_file, indent=4)

    return "Evaluation data has been created successfully"

await save_data()

Pinecone index already has 11671 vectors. Skipping ingestion.


'Evaluation data has been created successfully'

In [6]:
import time

def evaluate_system():
    data_file_path = os.path.join("tests", "data", "eval_data.json")
    with open(data_file_path, 'r') as json_file:
        eval_data = json.load(json_file)

    print(f"Dataset size: {len(eval_data)}")
    hf_dataset = Dataset.from_list(eval_data)

    # Single reliable LLM - no free tier rate limits
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)
    ragas_llm = LangchainLLMWrapper(llm)

    # Fast embeddings - no rate limits
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    emb = HuggingFaceEndpointEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
    ragas_emb = LangchainEmbeddingsWrapper(emb)
    
    # Only essential metrics - avoid rate limits
    scores = evaluate(
        hf_dataset,
        metrics=[
            faithfulness,
            context_precision,
        ],
        llm=ragas_llm,
        embeddings=ragas_emb,
        batch_size=5
    )
    return scores

print("Scores")
start = time.time()
result = evaluate_system()
print(f"Time: {time.time() - start:.1f}s")
print(result)

Scores
Dataset size: 14


C:\Users\Ziad\AppData\Local\Temp\ipykernel_36308\3661394236.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
C:\Users\Ziad\AppData\Local\Temp\ipykernel_36308\3661394236.py:19: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(emb)
Evaluating: 100%|██████████| 28/28 [04:14<00:00,  9.08s/it]

Time: 264.6s
{'faithfulness': 0.0000, 'context_precision': 0.5714}
